# Optimized Topic Modeling of AI Research Involving Nepali Authors

**ST7085CEM Advanced Machine Learning — Task 1**  
Bibek Paudyal (250288) · Siddhartha Bhatta (250620) · Sajan Mahat (250289)

End-to-end pipeline: corpus construction → preprocessing → three topic models →
comparative evaluation → thematic evolution.

---

### How this notebook relates to the repository

It **orchestrates** `src/nrtm/`; it does not re-implement it. Every function called here is the
same code the reported results came from, so the notebook cannot drift from the paper. If you want
to change the method, change the package — not a copy pasted into a cell.

Each step also exists as a numbered script (`scripts/01_…` … `scripts/11_…`). The notebook is for
reading and re-running the argument; the scripts are for reproducing it in one command.

### Runtime

| Mode | GA | Random control | Total |
|---|---|---|---|
| `QUICK_MODE = True` | ~3 min | ~3 min | **~15 min** |
| `QUICK_MODE = False` | ~69 min | ~66 min | **~2.5 h** |

Quick mode reduces the GA population and generations. It verifies the pipeline runs; it does **not**
reproduce the paper's numbers. Use `False` for that.

CPU only throughout — no GPU required.


## 1. Environment

Works in three places: a local clone, Google Colab, or any fresh machine. In Colab it clones the
repository (which carries the scraped corpus, ~14 MB) and installs the pinned requirements.


In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/bpdyl/ai-research-topic-modeling.git"
BRANCH   = "core-development"

IN_COLAB = "google.colab" in sys.modules

def find_repo_root(start: pathlib.Path) -> pathlib.Path | None:
    """Walk up looking for the package. Lets the notebook live in notebooks/ or the root."""
    for p in [start, *start.parents]:
        if (p / "src" / "nrtm").is_dir():
            return p
    return None

ROOT = find_repo_root(pathlib.Path.cwd())

if ROOT is None:
    print("Package not found locally — cloning...")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL], check=True)
    ROOT = pathlib.Path.cwd() / "ai-research-topic-modeling"

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("repo root :", ROOT)
print("colab     :", IN_COLAB)


In [ ]:
# Install only what is missing. Skipped entirely on a machine that already has the stack.
REQUIRED = ["gensim", "nltk", "pyyaml", "matplotlib", "seaborn", "wordcloud",
            "scikit-learn", "pandas", "scipy", "numpy"]
OPTIONAL = ["bertopic", "sentence-transformers", "umap-learn", "hdbscan", "torch"]

def missing(pkgs):
    import importlib.util
    alias = {"scikit-learn": "sklearn", "pyyaml": "yaml", "umap-learn": "umap",
             "sentence-transformers": "sentence_transformers"}
    return [p for p in pkgs if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]

need = missing(REQUIRED)
if need:
    print("installing:", need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

need_opt = missing(OPTIONAL)
if need_opt:
    print("installing (BERTopic stack):", need_opt)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need_opt], check=False)

print("environment ready")


In [ ]:
# BLAS thread caps must be set before numpy loads.
# gensim CoherenceModel spawns one worker per core; on Windows each child re-imports the whole
# scipy/BLAS stack, which exhausted the system commit limit during development. See DEC-015.
import nrtm  # applies the caps at import time

import nltk
for res in ["punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(res, quiet=True)

import time, json
import numpy as np
from nrtm.config import load_config, set_seed

cfg = load_config("config/default.yaml")
set_seed(cfg["seed"])

QUICK_MODE = True   # <-- set False to reproduce the paper

print(f"seed={cfg['seed']}  quick_mode={QUICK_MODE}")
print("corpus window:", cfg["corpus"]["year_min"], "-", cfg["corpus"]["year_max"])


## 2. Corpus

The scraper's export (`scraper/output/nepal_ai_papers.json`, 1,693 records) is **immutable raw
input**. This step applies documented filters and freezes the analysis corpus. Every later stage
reads the frozen file, so the corpus can only change by re-running this cell.

The `computer vision syndrome` exclusion is not arbitrary: the scraper's AI gate matches the phrase
*computer vision*, which also names an eye condition. 14 occupational-health papers entered the
corpus and formed their own topic at K=10. All were checked by hand (DEC-018).


In [ ]:
from nrtm.corpus.load import load_raw, to_documents, write_jsonl
from nrtm.corpus.filters import apply_filters
from nrtm.corpus.validate import build_report, format_report

records = load_raw(cfg.path("raw_corpus"))
docs = to_documents(records)
kept, report = apply_filters(docs, cfg)

print(f"{len(docs)} raw records\n\nInclusion funnel:")
running = report.n_in
for step in report.steps:
    print(f"  {step.name:<20} {running:>5} -> {step.kept:>5}  (-{step.dropped})")
    running = step.kept
print(f"  {'FINAL':<20} {report.n_out:>5}")

write_jsonl(kept, cfg.path("frozen_corpus"))
print("\n" + format_report(build_report(kept, cfg)))


## 3. Preprocessing — two branches

One shared cleaning stage, then **two different representations**:

- **LDA / GA-LDA** — tokens: lowercased, stop-word filtered, lemmatised, phrase-detected.
- **BERTopic** — cleaned *natural sentences*.

This split is not incidental. A sentence transformer depends on word order and function words;
giving it lemmatised bags of words would cripple the benchmark and make the central comparison
meaningless (OQ-015 / RISK-008).

Two preprocessing choices were made from measurement rather than convention (DEC-017):

1. **Domain stop words by document frequency.** `no_above=0.5` removed exactly one term ("model",
   59.8% DF); generic research verbs at 18–33% DF survived and would have appeared in every topic.
   77 terms at DF ≥ 5% were removed — but `level` and `value` were **kept**, because this corpus
   contains hydrology and health work where "water level" and "p-value" are content.
2. **Parenthesised acronym glosses stripped.** Authors write "artificial intelligence (AI)", and
   phrase detection then produced *both* `artificial_intelligence` (191 docs) and
   `artificial_intelligence_ai` (160) for one concept.


In [ ]:
from nrtm.corpus.load import read_jsonl
from nrtm.preprocessing.clean import clean_text
from nrtm.preprocessing.tokenize import tokenise_documents
from nrtm.preprocessing.represent import (
    build_phrases, build_dictionary, build_bow, build_tfidf, vocabulary_stats, write_jsonl_rows)

pre = cfg["preprocessing"]
frozen = read_jsonl(cfg.path("frozen_corpus"))
cleaned = [clean_text(d.text) for d in frozen]

tokens = tokenise_documents(
    cleaned,
    domain_stopwords=pre["domain_stopwords"],
    protected_acronyms=pre["protected_acronyms"],
    min_token_length=pre["min_token_length"])
tokens, _ = build_phrases(tokens, min_count=pre["phrase_min_count"],
                          threshold=pre["phrase_threshold"])

dcfg = pre["dictionary"]
dictionary, n_before = build_dictionary(tokens, **dcfg)
bow = build_bow(tokens, dictionary)
tfidf_model, tfidf = build_tfidf(bow)

stats = vocabulary_stats(tokens, dictionary)
print(f"types {n_before:,} -> dictionary {len(dictionary):,}")
print(f"tokens/doc: median {stats['tokens_per_doc']['median']}, empty docs {stats['empty_documents']}")

protected = [a.lower() for a in pre["protected_acronyms"]]
alive = [a for a in protected if a in dictionary.token2id]
print(f"acronyms retained {len(alive)}/{len(protected)}: {', '.join(alive)}")
print("\ntop terms:", ", ".join(t for t, _ in stats["most_common"][:15]))

# Persist so the scripts and the notebook share artefacts.
write_jsonl_rows([{"doc_id": d.doc_id, "year": d.year, "time_window": d.time_window,
                   "tokens": t} for d, t in zip(frozen, tokens)], cfg.path("tokens"))
write_jsonl_rows([{"doc_id": d.doc_id, "year": d.year, "time_window": d.time_window,
                   "text": c} for d, c in zip(frozen, cleaned)], cfg.path("bertopic_docs"))
dictionary.save(str(cfg.path("dictionary")))
from gensim.corpora import MmCorpus
MmCorpus.serialize(str(cfg.path("bow_corpus")), bow)
print("\nartefacts written")


## 4. Standard LDA baseline (EXP-001)

K is selected by C_v over a stated grid. The comparison the paper makes is therefore
**GA-search vs grid-search**, not GA vs an arbitrary hand-picked K.


In [ ]:
from nrtm.models.lda import fit_lda, sweep_num_topics, topic_top_words, perplexity
from nrtm.evaluation.coherence import all_coherences
from nrtm.evaluation.diversity import topic_diversity, mean_pairwise_jaccard
from nrtm.evaluation.stability import topic_stability

lcfg, ecfg = cfg["lda"], cfg["evaluation"]
seed = cfg["seed"]
fit_kwargs = dict(passes=lcfg["passes"], iterations=lcfg["iterations"],
                  chunksize=lcfg["chunksize"], alpha=lcfg["alpha"],
                  eta=lcfg.get("eta"), multicore=False)

k_values = list(range(lcfg["k_min"], lcfg["k_max"] + 1, lcfg["k_step"]))
if QUICK_MODE:
    k_values = [5, 10, 15, 20]

print(f"sweeping K over {k_values}")
sweep = sweep_num_topics(bow, dictionary, tokens, k_values=k_values, seed=seed,
                         top_n=ecfg["top_n_words"], coherence_measures=("c_v",),
                         processes=cfg["coherence_processes"], **fit_kwargs)

best_k = max(sweep, key=lambda r: r["c_v"])["k"]
print(f"\nselected K = {best_k}")

lda_model = fit_lda(bow, dictionary, num_topics=best_k, seed=seed, **fit_kwargs)
lda_topics = topic_top_words(lda_model, top_n=ecfg["top_n_words"])
lda_metrics = all_coherences(lda_topics, dictionary, texts=tokens, corpus=bow,
                             measures=tuple(ecfg["coherence_measures"]),
                             processes=cfg["coherence_processes"])
lda_metrics["diversity"] = topic_diversity(lda_topics, top_n=ecfg["diversity_top_n"])
lda_metrics["perplexity"] = perplexity(lda_model, bow)
lda_metrics["num_topics"] = best_k

stab_seeds = [seed + i for i in range(cfg["ga"]["stability_seeds"])]
lda_metrics["stability"] = topic_stability(
    lambda s: topic_top_words(fit_lda(bow, dictionary, num_topics=best_k, seed=s, **fit_kwargs),
                              top_n=ecfg["top_n_words"]),
    stab_seeds, top_n=ecfg["top_n_words"])["stability"]

for k, v in lda_metrics.items():
    print(f"  {k:<22} {v:.4f}" if isinstance(v, float) else f"  {k:<22} {v}")
print()
for i, t in enumerate(lda_topics):
    print(f"  {i:>2}: {', '.join(t)}")


In [ ]:
from nrtm.viz.figures import plot_coherence_vs_k
from IPython.display import Image, display

p = plot_coherence_vs_k(sweep, "figures/nb_coherence_vs_k.png", chosen_k=best_k)
display(Image(str(p)))


## 5. GA-Optimized LDA (EXP-002)

A hand-written genetic algorithm over *(K, α, η)*. Four design choices worth noting, all defended
in the paper:

- **Real-valued encoding**, not binary — α and η span two orders of magnitude.
- **Log-uniform initialisation** — uniform sampling would put ~90% of draws above 0.1 and barely
  explore the sparse-prior region.
- **Tournament selection** — fitness sits in a narrow band, so roulette would apply near-equal
  pressure to good and bad genomes.
- **BLX-α crossover** — plain interval crossover can only interpolate, so the population contracts
  toward its own mean and can never reach an optimum outside the initial spread.

Fitness caches on the rounded genome; the full run achieved a **39% hit rate**, cutting 900 LDA fits
to 549.


In [ ]:
from nrtm.models.ga import FitnessEvaluator, SearchSpace, run_ga

gcfg = cfg["ga"]
space = SearchSpace.from_config(gcfg)
ga_fit_kwargs = dict(passes=lcfg["passes"], iterations=lcfg["iterations"],
                     chunksize=lcfg["chunksize"], multicore=False)

pop, gens = gcfg["population_size"], gcfg["generations"]
if QUICK_MODE:
    pop, gens = 6, 3
    print("QUICK MODE — not the paper's numbers")

print(f"pop={pop} generations={gens} weights={gcfg['fitness_weights']}")

evaluator = FitnessEvaluator(
    corpus=bow, dictionary=dictionary, texts=tokens,
    weights=gcfg["fitness_weights"], stability_seeds=stab_seeds,
    top_n=ecfg["top_n_words"], diversity_top_n=ecfg["diversity_top_n"],
    coherence_processes=cfg["coherence_processes"], fit_kwargs=ga_fit_kwargs)

t0 = time.time()
ga_result = run_ga(evaluator, space, population_size=pop, generations=gens,
                   crossover_rate=gcfg["crossover_rate"], mutation_rate=gcfg["mutation_rate"],
                   n_elite=gcfg["elitism"], tournament_size=gcfg["tournament_size"], seed=seed)
print(f"\nGA finished in {time.time()-t0:.0f}s  {evaluator.stats()}")

g = ga_result.best.genome
print(f"best: K={g.k} alpha={g.alpha:.4f} eta={g.eta:.4f}  fitness={ga_result.best.fitness:.4f}")

ga_model = fit_lda(bow, dictionary, num_topics=g.k, seed=seed,
                   alpha=g.alpha, eta=g.eta, **ga_fit_kwargs)
ga_topics = topic_top_words(ga_model, top_n=ecfg["top_n_words"])
ga_metrics = all_coherences(ga_topics, dictionary, texts=tokens, corpus=bow,
                            measures=tuple(ecfg["coherence_measures"]),
                            processes=cfg["coherence_processes"])
ga_metrics["diversity"] = topic_diversity(ga_topics, top_n=ecfg["diversity_top_n"])
ga_metrics["perplexity"] = perplexity(ga_model, bow)
ga_metrics["num_topics"] = g.k
ga_metrics["stability"] = ga_result.best.stability

for k, v in ga_metrics.items():
    print(f"  {k:<22} {v:.4f}" if isinstance(v, float) else f"  {k:<22} {v}")


In [ ]:
from nrtm.viz.figures import plot_ga_convergence
p = plot_ga_convergence([h.to_dict() for h in ga_result.history],
                        "figures/nb_ga_convergence.png", baseline_cv=lda_metrics["c_v"])
display(Image(str(p)))

print("Watch the gap between best and mean. In the full run they nearly met by generation 8,")
print("meaning the population collapsed onto one genome and half the budget bought nothing.")


## 6. Random-search control (EXP-008) — the experiment that mattered most

The GA beat a **grid** of 8 K-values with fixed α and η, while itself searching a continuous 3-D
space. Two explanations fit that equally well: the evolutionary mechanism works, or fine search
beats coarse search and any method would do.

This control separates them. Identical fitness, seeds, LDA settings and candidate sampler; budget
matched on **unique** evaluations. One variable differs: how candidates are proposed.

**In the full run, random search won** — fitness 0.5282 vs 0.5196, C_v 0.5030 vs 0.4940. The paper
reports this as a negative result rather than omitting it (DEC-023).


In [ ]:
import random as _random

budget = evaluator.stats()["unique_evaluations"]   # match the GA exactly
print(f"random search, budget = {budget} unique evaluations")

rng = _random.Random(seed)
rs_eval = FitnessEvaluator(
    corpus=bow, dictionary=dictionary, texts=tokens,
    weights=gcfg["fitness_weights"], stability_seeds=stab_seeds,
    top_n=ecfg["top_n_words"], diversity_top_n=ecfg["diversity_top_n"],
    coherence_processes=cfg["coherence_processes"], fit_kwargs=ga_fit_kwargs)

best_rs = None
t0 = time.time()
for i in range(budget):
    r = rs_eval.evaluate(space.random_genome(rng))
    if best_rs is None or r.fitness > best_rs.fitness:
        best_rs = r
print(f"finished in {time.time()-t0:.0f}s")

rg = best_rs.genome
print(f"\nrandom best : K={rg.k} alpha={rg.alpha:.4f} eta={rg.eta:.4f}  fitness={best_rs.fitness:.4f}")
print(f"GA best     : K={g.k} alpha={g.alpha:.4f} eta={g.eta:.4f}  fitness={ga_result.best.fitness:.4f}")
delta = ga_result.best.fitness - best_rs.fitness
print(f"GA advantage: {delta:+.4f}")
print("\n" + ("GA ahead." if delta > 0.005 else
      "Random matched or beat the GA — do NOT claim evolutionary search is superior."))

rs_model = fit_lda(bow, dictionary, num_topics=rg.k, seed=seed,
                   alpha=rg.alpha, eta=rg.eta, **ga_fit_kwargs)
rs_topics = topic_top_words(rs_model, top_n=ecfg["top_n_words"])
rs_metrics = all_coherences(rs_topics, dictionary, texts=tokens, corpus=bow,
                            measures=tuple(ecfg["coherence_measures"]),
                            processes=cfg["coherence_processes"])
rs_metrics["diversity"] = topic_diversity(rs_topics, top_n=ecfg["diversity_top_n"])
rs_metrics["perplexity"] = perplexity(rs_model, bow)
rs_metrics["num_topics"] = rg.k
rs_metrics["stability"] = best_rs.stability


## 7. BERTopic benchmark (EXP-003)

Consumes the **natural-sentence** branch. Downloads `all-MiniLM-L6-v2` (~90 MB) on first run.

Two things are reported that are easy to omit and would flatter the model: the **outlier fraction**
(HDBSCAN leaves documents unclustered — 17.1% in the full run, so its coherence is computed over
83% of the corpus), and the **vocabulary overlap** with the shared scoring dictionary (66.4%).


In [ ]:
try:
    from nrtm.models.bertopic_model import (
        build_embeddings, fit_bertopic, bertopic_top_words, vocabulary_overlap)
    HAVE_BERTOPIC = True
except ImportError as e:
    print("BERTopic stack unavailable — skipping:", e)
    HAVE_BERTOPIC = False

bt_metrics, bt_topics = None, None
if HAVE_BERTOPIC:
    bcfg = cfg["bertopic"]
    import nltk as _nltk
    stops = set(_nltk.corpus.stopwords.words("english")) | set(pre["domain_stopwords"])

    docs_text = [c for c in cleaned]
    emb = build_embeddings(docs_text, bcfg["embedding_model"], show_progress=False)
    bt_model, assign, _p, info = fit_bertopic(
        docs_text, embeddings=emb, embedding_model=bcfg["embedding_model"],
        min_topic_size=bcfg["min_topic_size"], umap_kwargs=bcfg["umap"],
        hdbscan_kwargs=bcfg["hdbscan"], stopwords=sorted(stops), seed=seed)
    bt_topics = bertopic_top_words(bt_model, top_n=ecfg["top_n_words"])

    ov = vocabulary_overlap(bt_topics, dictionary)
    print(f"topics {info['n_topics']}  outliers {info['n_outliers']} "
          f"({100*info['outlier_fraction']:.1f}%)  vocab overlap {100*ov['coverage']:.1f}%")

    bt_metrics = all_coherences(bt_topics, dictionary, texts=tokens, corpus=bow,
                                measures=tuple(ecfg["coherence_measures"]),
                                processes=cfg["coherence_processes"])
    bt_metrics["diversity"] = topic_diversity(bt_topics, top_n=ecfg["diversity_top_n"])
    bt_metrics["num_topics"] = info["n_topics"]
    bt_metrics["outlier_fraction"] = info["outlier_fraction"]
    for i, t in enumerate(bt_topics):
        print(f"  {i:>2}: {', '.join(t)}")


## 8. Comparative evaluation (EXP-004)

**No model dominates** in the full run. That is the honest headline, and it is why the
interpretability column matters — it is the tiebreaker, and it *inverts* the coherence ranking.


In [ ]:
import pandas as pd

models = {"Standard LDA": lda_metrics, "GA-Optimized LDA": ga_metrics,
          "Random-Search LDA": rs_metrics}
if bt_metrics:
    models["BERTopic"] = bt_metrics

rows = ["num_topics", "c_v", "c_npmi", "u_mass", "diversity", "stability",
        "perplexity", "outlier_fraction"]
df = pd.DataFrame({name: {r: m.get(r) for r in rows} for name, m in models.items()})
display(df.round(4))

print("Caveats that must travel with this table:")
if bt_metrics and bt_metrics.get("outlier_fraction"):
    print(f"  - BERTopic leaves {100*bt_metrics['outlier_fraction']:.1f}% of documents unmodelled;")
    print("    LDA models 100%. The coherence figures are not over the same document set.")
print("  - The GA and random search both optimised C_v. Their C_v is not independent evidence;")
print("    c_npmi, u_mass and perplexity are, because neither strategy saw them.")


### Interpretability (EXP-007)

The one column that cannot be computed. Full-run values, from a **single LLM-as-judge rater** —
*not* the three independent human raters the proposal specifies, so inter-rater agreement is not
reported and the same system fitted the models it rated (DEC-020).

| Model | Interpretability |
|---|---:|
| BERTopic | **4.64** |
| Standard LDA | 3.10 |
| GA-Optimized LDA | **2.71** (last) |

**This inverts the coherence ranking.** The GA converged to K=7 because fewer, broader topics
scored well on coherence + diversity + stability — and broader topics are less interpretable.
Optimising coherence metrics worked *against* the thing we actually wanted.


## 9. Thematic evolution (EXP-005)

One global model, proportions normalised **within** window so window size cancels. The 2015–17
window holds 8 documents — reported, but excluded from trend fitting. Per-window *n* appears on
every figure, because a proportion from 8 documents must not be read like one from 674.


In [ ]:
from nrtm.models.lda import doc_topic_matrix
from nrtm.temporal.proportions import window_proportions, topic_trends
from nrtm.viz.figures import plot_topic_trends

windows = [f"{a}-{b}" for a, b in cfg["corpus"]["time_windows"]]
doc_windows = [d.time_window for d in frozen]

dt = doc_topic_matrix(ga_model, bow, num_topics=g.k)
props, counts = window_proportions(dt, windows, doc_windows, normalize_within_window=True)
trends, excluded = topic_trends(props, windows, counts, min_docs=30)
labels = [", ".join(t[:3]) for t in ga_topics]

print("documents per window:", dict(zip(windows, counts)))
if excluded:
    print("excluded from trend fitting:", excluded)
print()
for tr in trends:
    series = "".join(f"{(tr['by_window'][w] or float('nan')):>10.3f}" for w in windows)
    print(f"{labels[tr['topic']][:34]:<36}{series}{tr['trend']:>12}")

p = plot_topic_trends(props, windows, counts, labels, "figures/nb_trends.png",
                      top_n=min(8, props.shape[1]))
display(Image(str(p)))


## 10. Corpus figures


In [ ]:
from nrtm.viz.figures import plot_wordcloud, plot_cooccurrence_network, plot_corpus_by_year

display(Image(str(plot_corpus_by_year([d.year for d in frozen], "figures/nb_years.png"))))
display(Image(str(plot_wordcloud(tokens, "figures/nb_wordcloud.png"))))
display(Image(str(plot_cooccurrence_network(tokens, "figures/nb_cooccurrence.png"))))


## 11. What the full run found

Reference values from the reported experiments (`.brain/EXPERIMENT_LOG.md`, EXP-001 … EXP-008).
Quick mode will not match these.

| Metric | Standard LDA | GA-LDA | Random-Search LDA | BERTopic |
|---|---:|---:|---:|---:|
| Topics (K) | 10 | 7 | 8 | 14 |
| C_v | 0.4483 | 0.4940 | 0.5030 | **0.5107** |
| c_npmi | −0.0230 | 0.0112 | **0.0245** | −0.0300 |
| u_mass | −3.3586 | **−2.1796** | −2.5477 | −3.4444 |
| Diversity | 0.7900 | **0.8000** | **0.8000** | 0.7286 |
| Stability | 0.2526 | 0.3375 | 0.3382 | **0.8418** |
| Perplexity | 172.1 | 167.8 | **165.9** | — |
| Unmodelled | **0%** | **0%** | **0%** | 17.1% |
| Interpretability | 3.10 | 2.71 | [TODO] | **4.64** |

### Three findings

1. **GA-LDA beats the grid baseline on every metric**, including three it never optimised.
2. **But random search beats the GA at equal budget.** The gain comes from searching *(K, α, β)*
   finely, not from evolution. Only 1 of 183 random draws beat the GA, so the GA behaves as
   variance reduction rather than optimum finding.
3. **Interpretability inverts the coherence ranking.** The model that won most numeric metrics is
   the least interpretable.

On thematic evolution, GA-LDA and BERTopic **independently agree**: generative AI in education is
the dominant rising theme (+0.131 and +0.181 share), and classical computer-vision themes decline.
A finding that survives two model families is not an artefact of either.

---

### Reproducing this without the notebook

```bash
pip install -r requirements.txt
python -c "import nltk; [nltk.download(r) for r in ['punkt_tab','stopwords','wordnet','omw-1.4']]"
for s in scripts/0*.py scripts/1*.py; do python "$s"; done
```

Each script writes `results/runs/<RUN_ID>/` with the exact config, the git commit and its metrics,
so any number here is traceable to the run that produced it. Seeds are fixed at 42 throughout.

Full context, decisions and limitations: `.brain/` (not published — it contains candid
grading-risk notes) and `paper/paper_draft.md`.
